In [1]:
import numpy as np
import pandas as pd

raw = pd.read_csv(
    "../data/raw/supernova/Pantheon+SH0ES.dat",
    sep=r"\s+",
    dtype={"CID": "string"},
)

cov_file = np.loadtxt(
    "../data/raw/supernova/Pantheon+SH0ES_STAT+SYS.cov"
)

n = int(cov_file[0])
cov = cov_file[1:].reshape(n, n)

assert n == len(raw)

raw[["CID", "zHD", "zHEL", "m_b_corr"]].head()

,CID,zHD,zHEL,m_b_corr
0,2011fe,0.00122,0.00082,9.74571
1,2011fe,0.00122,0.00082,9.80286
2,2012cg,0.00256,0.00144,11.47030
3,2012cg,0.00256,0.00144,11.49190
4,1994DRichmond,0.00299,0.00187,11.52270


In [2]:
z_min = 0.01

valid = np.isfinite(raw[["zHD", "zHEL", "m_b_corr"]].values).all(axis=1)
mask = valid & (raw["zHD"].values > z_min) & (raw["zHEL"].values > -1)

sn = raw.loc[mask, ["CID", "zHD", "zHEL", "m_b_corr"]]
cov = cov[mask][:, mask]

assert np.allclose(cov, cov.T, rtol=1e-5, atol=1e-7)
cov = (cov + cov.T) / 2

pd.DataFrame({
    "원본 관측 수": [len(raw)],
    "선택 관측 수": [len(sn)],
    "z_min": [z_min],
})

,원본 관측 수,선택 관측 수,z_min
0,1701,1590,0.01


In [3]:
sn.to_csv("processed/sn.csv", index=False)

np.savez_compressed(
    "processed/sn_covariance.npz",
    covariance=cov,
)

pd.DataFrame({
    "z_min": [z_min],
    "covariance": ["STAT+SYS"],
    "n_observations": [len(sn)],
}).to_csv("processed/selection.csv", index=False)